# The Machine-Translation Dataset

In [1]:
import os
import sys
import pickle
import torch
import gc
import torch.optim as optim
import torch.nn as nn
from torchtext.vocab import GloVe

PROJECT_DIR = os.path.expanduser("~/Vietnamese-Poem-Generation/")
TRANSLATION_UTILS_DIR = os.path.join(PROJECT_DIR, "translation/utils")
TRANSLATION_MODELS_DIR = os.path.join(PROJECT_DIR, "translation/models")
sys.path.append(PROJECT_DIR)
sys.path.append(TRANSLATION_UTILS_DIR)
sys.path.append(TRANSLATION_MODELS_DIR)
from constants import *

from data_splitting import split_and_save_data
from tokenization import build_vocabulary
from data_loader import *
from network import *
from train import train_model
from evaluation import *

2025-02-16 09:19:57.172079: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739672397.243698    6112 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739672397.265372    6112 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-16 09:19:57.440459: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Overview

In [2]:
en_data_dir = os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.en")
vi_data_dir = os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.vi")

In [3]:
en_data = open(en_data_dir, "r").readlines()
vi_data = open(vi_data_dir, "r").readlines()

In [4]:
en_data = [line.strip() for line in en_data]
vi_data = [line.strip() for line in vi_data]

In [5]:
(len(en_data), len(vi_data))

(326417, 326417)

In [6]:
en_data[100], vi_data[100]

("And if they judge how much they're going to get paid on your capital that they've invested, based on the short-term returns, you're going to get short-term decisions.",
 'Và nếu họ xem xét việc họ sẽ được trả bao nhiêu tiền dựa trên số vốn của bạn mà họ đã đầu tư, dựa trên những món hoàn trả ngắn hạn, bạn sẽ nhận được những quyết định ngắn hạn.')

In [7]:
max_en_len = max(len(line.split()) for line in en_data)
max_vi_len = max(len(line.split()) for line in vi_data)
print("Max English sentence length:", max_en_len, "\nMax Vietnamese sentence length:", max_vi_len)

Max English sentence length: 625 
Max Vietnamese sentence length: 838


In [8]:
# (OPTIONAL)
# Get 20000 english sentences and their corresponding vietnamese sentences
en_data = en_data[:20000]
vi_data = vi_data[:20000]

# Save the data under .txt format
with open(os.path.join(TRANSLATION_DATA_DIR, "20K_TED2020.en-vi.en"), "w") as f:
    f.write("\n".join(en_data))

with open(os.path.join(TRANSLATION_DATA_DIR, "20K_TED2020.en-vi.vi"), "w") as f:
    f.write("\n".join(vi_data))

# Print length of the data
print((len(en_data), len(vi_data)))

(20000, 20000)


## Data Splitting

In [2]:
split_and_save_data(
    source_file=os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.vi"),
    target_file=os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.en")
)

# split_and_save_data(
#     source_file=os.path.join(TRANSLATION_DATA_DIR, "20K_TED2020.en-vi.vi"),
#     target_file=os.path.join(TRANSLATION_DATA_DIR, "20K_TED2020.en-vi.en")
# )

Dataset successfully split!
Train: 261133 samples, Val: 32641 samples, Test: 32643 samples.


## Tokenization

In [8]:
vocab_transform = build_vocabulary(
    source_file=os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.vi"),
    target_file=os.path.join(TRANSLATION_DATA_DIR, "TED2020.en-vi.en")
)

In [5]:
len(vocab_transform[SRC_LANGUAGE].get_itos()), len(vocab_transform[TGT_LANGUAGE].get_itos())

(91313, 74479)

In [6]:
vocab_transform[SRC_LANGUAGE].get_itos()[:10], vocab_transform[TGT_LANGUAGE].get_itos()[:10]

(['<unk>', '<pad>', '<bos>', '<eos>', ',', '.', 'là', 'và', 'một', 'tôi'],
 ['<unk>', '<pad>', '<bos>', '<eos>', ',', '.', 'the', 'and', 'to', "'"])

Save the vocabulary:

In [ ]:
with open(os.path.join(STORAGE_DIR, "translation_vocab_transform.pkl"), "wb") as f:
    pickle.dump(vocab_transform, f)

Load the vocabulary:

In [2]:
with open(os.path.join(STORAGE_DIR, "translation_vocab_transform.pkl"), "rb") as f:
    vocab_transform = pickle.load(f)

## Data Loader

In [3]:
train_loader = get_dataloader(
    source_file=os.path.join(TRANSLATION_TRAIN_DIR, "train.vi"),
    target_file=os.path.join(TRANSLATION_TRAIN_DIR, "train.en"),
    vocab_transform=vocab_transform,
    batch_size=16,
    mode="train"
)

val_loader = get_dataloader(
    source_file=os.path.join(TRANSLATION_VAL_DIR, "val.vi"),
    target_file=os.path.join(TRANSLATION_VAL_DIR, "val.en"),
    vocab_transform=vocab_transform,
    batch_size=8,
    mode="val"
)

test_loader = get_dataloader(
    source_file=os.path.join(TRANSLATION_TEST_DIR, "test.vi"),
    target_file=os.path.join(TRANSLATION_TEST_DIR, "test.en"),
    vocab_transform=vocab_transform,
    batch_size=8,
    mode="test"
)

In [4]:
len(train_loader), len(val_loader), len(test_loader)

(16321, 4081, 4081)

In [5]:
src_ids, tgt_ids = next(iter(train_loader))
src_ids.shape, tgt_ids.shape

(torch.Size([16, 48]), torch.Size([16, 53]))

In [7]:
src_ids, tgt_ids

(tensor([[   2,    0,   73, 2626,    4,    0,  176,   18,    0,    4,    6,   18,
             0,    0,  569,    0,    4,   38,    0,   34,   57,   88,   31,  329,
             0,    5,    3,    1,    1,    1,    1,    1,    1,    1,    1,    1,
             1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
             1,    1,    1,    1,    1,    1],
         [   2,    0,   14,    0,    0,   16,  194, 1004,    4,   35,   75,   16,
            79,    5,    3,    1,    1,    1,    1,    1,    1,    1,    1,    1,
             1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
             1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
             1,    1,    1,    1,    1,    1],
         [   2,    0,    8,   19,   86,   22,  950,   13,    0,    4,   16,   64,
           102,  349,    0,    4,    0,  349,    0,   51,   66,    0,   32,   30,
          1590,    0,    8,   16,   24,    9,   10,    0,    5,    3,    1,    1,
    

## Training

In [6]:
input_size = len(vocab_transform[SRC_LANGUAGE])
output_size = len(vocab_transform[TGT_LANGUAGE])
hidden_size = 300

In [10]:
print(f"Input size: {input_size}, Output size: {output_size}, Hidden size: {hidden_size}")

Input size: 91313, Output size: 74479, Hidden size: 300


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [6]:
glove = GloVe(name="6B", dim=300)  # Use 300-dimensional GloVe embedding
pretrained_embedding = glove.vectors

model = Seq2Seq_GRU(
    encoder=EncoderGRU(input_size, hidden_size, pretrained_embedding=pretrained_embedding, freeze_embedding=True),
    decoder=DecoderGRU(hidden_size, output_size, pretrained_embedding=pretrained_embedding, freeze_embedding=True),
    device=device
).to(device)

In [8]:
optimizer = optim.Adam(model.parameters())
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=10)

In [ ]:
torch.cuda.empty_cache()
gc.collect()

train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=1,
    device=device,
    model_name="seq2seq_gru",
)

## Testing

In [8]:
glove = GloVe(name="6B", dim=300)  # Use 300-dimensional GloVe embedding
pretrained_embedding = glove.vectors

# Adjust output_size to match the checkpoint
checkpoint = torch.load(os.path.join(STORAGE_DIR, "seq2seq_gru.pt"))
output_size = checkpoint['decoder.out.weight'].size(0)

model = Seq2Seq_GRU(
    encoder=EncoderGRU(input_size, hidden_size, pretrained_embedding=pretrained_embedding, freeze_embedding=True),
    decoder=DecoderGRU(hidden_size, output_size, pretrained_embedding=pretrained_embedding, freeze_embedding=True),
    device=device
).to(device)

model.load_state_dict(checkpoint)

<All keys matched successfully>

In [10]:
print("Evaluating on test_loader...")
test_model(model, test_loader, vocab_transform, device)

Evaluating on test_loader...
Example 1:
Input:      Trượt cái màu xanh ra khỏi đường đi .
Reference:  well , move the blue one out of the way .
Prediction: <bos> the the the the the . .
--------------------------------------------------
Example 2:
Input:      Giờ làm nó khó hơn chút . Nhưng vẫn rất dễ .
Reference:  here , let ' s make it a little harder . still pretty easy .
Prediction: <bos> now , it ' s hard to make it better .
--------------------------------------------------
Example 3:
Input:      Bây giờ làm nó khó hơn 1 chút , chút nữa .
Reference:  now we ' ll make it harder , a little harder .
Prediction: <bos> now , it ' s hard to make it a better .
--------------------------------------------------
Example 4:
Input:      Và bây giờ cái này thì hơi khó nhằn .
Reference:  now , this one is a little bit trickier .
Prediction: <bos> and this is is this . .
--------------------------------------------------
Example 5:
Input:      Bạn biết hả ? Làm cái gì ở đây ?
Reference:  you k

KeyboardInterrupt: 

In [9]:
sentences = [
    "anh yêu em, Vân Anh",
    "Anh nhớ em nhiều lắm"
]

for sentence in sentences:
    print(f"Vietnamese: {sentence}")
    translated_sentence = translate_sentence(sentence, model, vocab_transform, device, max_len=10)
    print(f"English: {translated_sentence}")
    print("=" * 50)

Vietnamese: anh yêu em, Vân Anh
English: <bos> he said , you ' s .
Vietnamese: Anh nhớ em nhiều lắm
English: <bos> he ' s too much .


# The Poem Dataset